# Phase 2: R:R Gate Validation

**Goal:** Match Pine script's stop calculation and minimum R:R filter exactly

## The Bug We're Fixing
On 2025-10-30, Pine showed: `SKIP: 1.2:1 < 1.5 min`  
But the Python backtester took the trade and lost -1R.

## Pine Logic (Source of Truth)

```pine
// 1. Calculate stops for each type
atrStop = entry - (ATR * atrMultiplier)  // default 2.0
swingStop = swingLow - (ATR * swingBuffer)  // RTH only
vwapStop = vwapValue - (ATR * vwapStopDistance)

// 2. Calculate achievable R:R for each
risk = |entry - stop|
targetATRs = (risk * rrDesired) / ATR
if targetATRs > maxTargetATR:
    bestRR = (maxTargetATR * ATR) / risk  // Cap it
else:
    bestRR = rrDesired

// 3. Auto-select best stop (highest R:R)
// 4. THE GATE
tradeFeasible = finalRR >= minAcceptableRR  // 1.5
```

In [1]:
# Cell 1: Setup
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')

import pandas as pd
import numpy as np
from dotenv import load_dotenv
load_dotenv(r'C:\Users\phemm\orb_lab\.env')

from data_collector import PolygonDataCollector

print("✓ Setup complete")

✓ Setup complete


In [2]:
# Cell 2: Fetch data and run Phase 1 (ORB detection)
collector = PolygonDataCollector()
df = collector.fetch_bars('NVDA', days_back=60, bar_size=1)

# === Phase 1 functions (already validated) ===
def detect_orb_session(df, orb_minutes=5):
    def is_in_orb(ts):
        bar_total = ts.hour * 60 + ts.minute
        orb_start = 9 * 60 + 30  # 9:30
        orb_end = orb_start + orb_minutes
        return orb_start <= bar_total < orb_end
    
    df = df.copy()
    df['in_session'] = df.index.map(is_in_orb)
    df['is_new_session'] = df['in_session'] & ~df['in_session'].shift(1, fill_value=False)
    return df

def calculate_orb_levels(df):
    df = df.copy()
    df['orb_high'] = np.nan
    df['orb_low'] = np.nan
    
    current_high, current_low = np.nan, np.nan
    for i in range(len(df)):
        if df['is_new_session'].iloc[i]:
            current_high = df['high'].iloc[i]
            current_low = df['low'].iloc[i]
        elif df['in_session'].iloc[i]:
            current_high = max(current_high, df['high'].iloc[i])
            current_low = min(current_low, df['low'].iloc[i])
        df.iloc[i, df.columns.get_loc('orb_high')] = current_high
        df.iloc[i, df.columns.get_loc('orb_low')] = current_low
    return df

def calc_atr(df, period=14):
    df = df.copy()
    tr1 = df['high'] - df['low']
    tr2 = abs(df['high'] - df['close'].shift(1))
    tr3 = abs(df['low'] - df['close'].shift(1))
    df['tr'] = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
    df['atr'] = df['tr'].ewm(alpha=1/period, adjust=False).mean()
    return df

# Apply Phase 1
df = detect_orb_session(df)
df = calculate_orb_levels(df)
df['orb_complete'] = ~df['in_session'] & df['orb_high'].notna() & df['orb_low'].notna()
df = calc_atr(df)

print(f"✓ Loaded {len(df)} bars with ORB detection")

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching NVDA (1-min bars, 60 days)...
  ✓ Loading from cache (0.9 hours old)
✓ Loaded 16064 bars with ORB detection


In [3]:
# Cell 3: VWAP Calculation (session-based, matching Pine)

def calc_session_vwap(df):
    """
    Pine: ta.vwap(hlc3) but resets each session
    For simplicity, we use daily VWAP reset at 9:30
    """
    df = df.copy()
    df['typical_price'] = (df['high'] + df['low'] + df['close']) / 3
    df['tp_volume'] = df['typical_price'] * df['volume']
    
    # Detect new day (9:30 bar)
    df['new_day'] = (df.index.hour == 9) & (df.index.minute == 30)
    df['day_group'] = df['new_day'].cumsum()
    
    # Cumulative sums within each day
    df['cum_tp_vol'] = df.groupby('day_group')['tp_volume'].cumsum()
    df['cum_vol'] = df.groupby('day_group')['volume'].cumsum()
    
    df['vwap'] = df['cum_tp_vol'] / df['cum_vol']
    
    return df

df = calc_session_vwap(df)
print("✓ VWAP calculated")

✓ VWAP calculated


In [4]:
# Cell 4: Stop Loss Calculations - MATCHING PINE EXACTLY

def find_swing_low_rth(df, idx, lookback=5):
    """
    Pine: findSwingLowRegularHours(lookback)
    Only considers RTH bars (9:30-16:00)
    """
    swing_low = None
    valid_bars = 0
    
    for i in range(1, lookback * 3 + 1):
        if idx - i < 0:
            break
        bar_time = df.index[idx - i]
        # Check if RTH
        bar_minutes = bar_time.hour * 60 + bar_time.minute
        if 570 <= bar_minutes < 960:  # 9:30 to 16:00
            bar_low = df['low'].iloc[idx - i]
            if swing_low is None or bar_low < swing_low:
                swing_low = bar_low
            valid_bars += 1
            if valid_bars >= lookback:
                break
    
    if swing_low is None:
        swing_low = df['low'].iloc[idx - 1] if idx > 0 else df['low'].iloc[idx]
    
    return swing_low


def find_swing_high_rth(df, idx, lookback=5):
    """Pine: findSwingHighRegularHours(lookback)"""
    swing_high = None
    valid_bars = 0
    
    for i in range(1, lookback * 3 + 1):
        if idx - i < 0:
            break
        bar_time = df.index[idx - i]
        bar_minutes = bar_time.hour * 60 + bar_time.minute
        if 570 <= bar_minutes < 960:  # RTH
            bar_high = df['high'].iloc[idx - i]
            if swing_high is None or bar_high > swing_high:
                swing_high = bar_high
            valid_bars += 1
            if valid_bars >= lookback:
                break
    
    if swing_high is None:
        swing_high = df['high'].iloc[idx - 1] if idx > 0 else df['high'].iloc[idx]
    
    return swing_high


def calculate_all_stops(df, idx, is_long, 
                        atr_mult=2.0, swing_lookback=5, swing_buffer=0.02,
                        vwap_distance=0.3, hybrid_exclude_vwap=False):
    """
    Pine: calculateStopLoss(isLong, entryPrice, stopType)
    Returns dict with all stop types and their prices
    """
    entry = df['close'].iloc[idx]
    atr = df['atr'].iloc[idx]
    vwap = df['vwap'].iloc[idx]
    
    # ATR Stop
    if is_long:
        atr_stop = entry - (atr * atr_mult)
    else:
        atr_stop = entry + (atr * atr_mult)
    
    # Swing Stop
    if is_long:
        swing_low = find_swing_low_rth(df, idx, swing_lookback)
        swing_stop = swing_low - (atr * swing_buffer)
        swing_stop = min(swing_stop, entry - 0.01)  # Must be below entry
    else:
        swing_high = find_swing_high_rth(df, idx, swing_lookback)
        swing_stop = swing_high + (atr * swing_buffer)
        swing_stop = max(swing_stop, entry + 0.01)  # Must be above entry
    
    # VWAP Stop
    if is_long:
        vwap_stop = vwap - (atr * vwap_distance)
        vwap_stop = min(vwap_stop, entry - 0.01)
    else:
        vwap_stop = vwap + (atr * vwap_distance)
        vwap_stop = max(vwap_stop, entry + 0.01)
    
    # Hybrid Stop (most conservative = closest to entry)
    if is_long:
        if hybrid_exclude_vwap:
            hybrid_stop = max(atr_stop, swing_stop)
        else:
            hybrid_stop = max(atr_stop, swing_stop, vwap_stop)
    else:
        if hybrid_exclude_vwap:
            hybrid_stop = min(atr_stop, swing_stop)
        else:
            hybrid_stop = min(atr_stop, swing_stop, vwap_stop)
    
    return {
        'entry': entry,
        'atr': atr,
        'vwap': vwap,
        'atr_stop': atr_stop,
        'swing_stop': swing_stop,
        'vwap_stop': vwap_stop,
        'hybrid_stop': hybrid_stop
    }

print("✓ Stop calculation functions defined")

✓ Stop calculation functions defined


In [5]:
# Cell 5: R:R Calculation and Gate - THE KEY LOGIC

def calculate_best_rr(stops, is_long, rr_desired=2.0, max_target_atr=3.0):
    """
    Pine: evaluateBestStop(isLong, entryPrice, atrVal, rrDesired)
    
    Calculates achievable R:R for each stop type.
    If target distance > maxTargetATR, R:R is capped.
    """
    entry = stops['entry']
    atr = stops['atr']
    
    results = {}
    
    for stop_type in ['atr_stop', 'swing_stop', 'vwap_stop', 'hybrid_stop']:
        stop_price = stops[stop_type]
        risk = abs(entry - stop_price)
        
        if risk <= 0:
            results[stop_type] = {'stop': stop_price, 'risk': 0, 'rr': 0}
            continue
        
        # Target at desired R:R
        target_distance = risk * rr_desired
        target_atrs = target_distance / atr
        
        # Cap R:R if target too far
        if target_atrs > max_target_atr:
            best_rr = (max_target_atr * atr) / risk
        else:
            best_rr = rr_desired
        
        results[stop_type] = {
            'stop': stop_price,
            'risk': risk,
            'target_atrs': target_atrs,
            'rr': best_rr
        }
    
    return results


def select_best_stop(rr_results):
    """
    Pine: Selects stop type with highest achievable R:R
    """
    best_type = None
    best_rr = 0
    
    for stop_type, data in rr_results.items():
        if data['rr'] > best_rr:
            best_rr = data['rr']
            best_type = stop_type
    
    return best_type, best_rr


def check_rr_gate(final_rr, min_acceptable_rr=1.5):
    """
    Pine: tradeFeasible = finalRR >= minAcceptableRR
    """
    return final_rr >= min_acceptable_rr

print("✓ R:R gate functions defined")

✓ R:R gate functions defined


In [6]:
# Cell 6: Test on 2025-10-30 (the SKIP trade)

test_date = '2025-10-30'
day_df = df[df.index.date.astype(str) == test_date].copy()

print(f"=== Testing R:R Gate on {test_date} ===")
print(f"\nORB High: {day_df['orb_high'].iloc[-1]:.2f}")
print(f"ORB Low:  {day_df['orb_low'].iloc[-1]:.2f}")

# Find the first bar that breaks below ORB Low (potential SHORT)
breakout_threshold = 0.1  # ATR multiplier

for i, (ts, row) in enumerate(day_df.iterrows()):
    if not row['orb_complete']:
        continue
    
    orb_low = row['orb_low']
    threshold = row['atr'] * breakout_threshold
    
    # Check for SHORT breakout
    if row['close'] < orb_low - threshold:
        idx = day_df.index.get_loc(ts)
        print(f"\n📍 SHORT breakout detected at {ts}")
        print(f"   Close: {row['close']:.2f}")
        print(f"   ORB Low - threshold: {orb_low - threshold:.2f}")
        
        # Calculate all stops
        # Need to find actual index in full df
        full_idx = df.index.get_loc(ts)
        stops = calculate_all_stops(df, full_idx, is_long=False)
        
        print(f"\n   === Stop Prices ===")
        print(f"   Entry: {stops['entry']:.2f}")
        print(f"   ATR:   {stops['atr']:.4f}")
        print(f"   ATR Stop:    {stops['atr_stop']:.2f}")
        print(f"   Swing Stop:  {stops['swing_stop']:.2f}")
        print(f"   VWAP Stop:   {stops['vwap_stop']:.2f}")
        print(f"   Hybrid Stop: {stops['hybrid_stop']:.2f}")
        
        # Calculate R:R for each
        rr_results = calculate_best_rr(stops, is_long=False)
        
        print(f"\n   === R:R by Stop Type ===")
        for stop_type, data in rr_results.items():
            print(f"   {stop_type}: Risk=${data['risk']:.2f}, R:R={data['rr']:.2f}")
        
        # Select best
        best_type, best_rr = select_best_stop(rr_results)
        print(f"\n   Best Stop: {best_type} with R:R = {best_rr:.2f}")
        
        # THE GATE
        min_rr = 1.5
        passes_gate = check_rr_gate(best_rr, min_rr)
        
        print(f"\n   === THE GATE ===")
        print(f"   Final R:R: {best_rr:.2f}")
        print(f"   Min Required: {min_rr}")
        if passes_gate:
            print(f"   ✓ PASSES - Take trade")
        else:
            print(f"   ✗ FAILS - SKIP: {best_rr:.2f}:1 < {min_rr} min")
        
        break  # Just check first breakout

=== Testing R:R Gate on 2025-10-30 ===

ORB High: 206.16
ORB Low:  202.87

📍 SHORT breakout detected at 2025-10-30 09:35:00-04:00
   Close: 202.35
   ORB Low - threshold: 202.80

   === Stop Prices ===
   Entry: 202.35
   ATR:   0.7208
   ATR Stop:    203.80
   Swing Stop:  206.17
   VWAP Stop:   204.38
   Hybrid Stop: 203.80

   === R:R by Stop Type ===
   atr_stop: Risk=$1.44, R:R=1.50
   swing_stop: Risk=$3.82, R:R=0.57
   vwap_stop: Risk=$2.02, R:R=1.07
   hybrid_stop: Risk=$1.44, R:R=1.50

   Best Stop: atr_stop with R:R = 1.50

   === THE GATE ===
   Final R:R: 1.50
   Min Required: 1.5
   ✗ FAILS - SKIP: 1.50:1 < 1.5 min


In [7]:
# Cell 7: Compare with TradingView

print("""
=== VALIDATION CHECKLIST ===

Open TradingView for NVDA on 2025-10-30.
Look at the SKIP label on the SHORT signal.

Pine shows: "SKIP: 1.2:1 < 1.5 min"

Does Python calculate the same R:R? ______

If yes → Phase 2 R:R gate is accurate ✓
If different → Need to debug stop calculation
""")


=== VALIDATION CHECKLIST ===

Open TradingView for NVDA on 2025-10-30.
Look at the SKIP label on the SHORT signal.

Pine shows: "SKIP: 1.2:1 < 1.5 min"

Does Python calculate the same R:R? ______

If yes → Phase 2 R:R gate is accurate ✓
If different → Need to debug stop calculation



In [8]:
# Cell 8: Test on 2025-10-29 (the trade that DID execute)

test_date = '2025-10-29'
day_df = df[df.index.date.astype(str) == test_date].copy()

print(f"=== Testing R:R Gate on {test_date} (should PASS) ===")
print(f"\nORB High: {day_df['orb_high'].iloc[-1]:.2f}")
print(f"ORB Low:  {day_df['orb_low'].iloc[-1]:.2f}")

breakout_threshold = 0.1

for i, (ts, row) in enumerate(day_df.iterrows()):
    if not row['orb_complete']:
        continue
    
    orb_high = row['orb_high']
    threshold = row['atr'] * breakout_threshold
    
    # Check for LONG breakout
    if row['close'] > orb_high + threshold:
        full_idx = df.index.get_loc(ts)
        print(f"\n📍 LONG breakout detected at {ts}")
        print(f"   Close: {row['close']:.2f}")
        
        stops = calculate_all_stops(df, full_idx, is_long=True)
        rr_results = calculate_best_rr(stops, is_long=True)
        
        print(f"\n   === R:R by Stop Type ===")
        for stop_type, data in rr_results.items():
            print(f"   {stop_type}: Risk=${data['risk']:.2f}, R:R={data['rr']:.2f}")
        
        best_type, best_rr = select_best_stop(rr_results)
        passes_gate = check_rr_gate(best_rr, 1.5)
        
        print(f"\n   === THE GATE ===")
        print(f"   Best: {best_type} with R:R = {best_rr:.2f}")
        print(f"   {'✓ PASSES' if passes_gate else '✗ FAILS'}")
        
        break

=== Testing R:R Gate on 2025-10-29 (should PASS) ===

ORB High: 211.08
ORB Low:  207.09

📍 LONG breakout detected at 2025-10-29 09:36:00-04:00
   Close: 211.32

   === R:R by Stop Type ===
   atr_stop: Risk=$1.79, R:R=1.50
   swing_stop: Risk=$3.69, R:R=0.73
   vwap_stop: Risk=$2.08, R:R=1.29
   hybrid_stop: Risk=$1.79, R:R=1.50

   === THE GATE ===
   Best: atr_stop with R:R = 1.50
   ✓ PASSES


In [10]:
# Check the 09:35 bar on 10/30
bar = df.loc['2025-10-30 09:35:00-04:00']
print(f"Open:  {bar['open']:.2f}")
print(f"High:  {bar['high']:.2f}")
print(f"Low:   {bar['low']:.2f}")
print(f"Close: {bar['close']:.2f}")

Open:  202.98
High:  203.28
Low:   202.35
Close: 202.35


In [12]:
# Compare ATR - might be the culprit
ts = '2025-10-30 09:35:00-04:00'
print(f"Python ATR: {df.loc[ts, 'atr']:.4f}")

# What ATR would give us 1.2 R:R?
# R:R = (maxTargetATR * ATR) / risk
# 1.2 = (3.0 * ATR) / 1.44
# ATR = 1.2 * 1.44 / 3.0 = 0.576
print(f"ATR needed for 1.2 R:R: 0.576")

Python ATR: 0.7208
ATR needed for 1.2 R:R: 0.576


In [13]:
# Cell: CORRECTED Volatility State Calculation (matching Pine exactly)

def calc_vol_state_pine(df, fast_atr_len=5, 
                        low_thresh=0.8, high_thresh=1.3, extreme_thresh=2.0):
    """
    Pine's EXACT volatility factor calculation:
    
    volFactor = sessionVF × htfVF
    
    sessionVF = orbATR_fast / orbSessionBaseline
    htfVF = dailyATR / dailyATRslow
    """
    df = df.copy()
    
    # --- Component 1: Session Volatility Factor ---
    
    # Fast ATR (5-period by default)
    tr = pd.concat([
        df['high'] - df['low'],
        abs(df['high'] - df['close'].shift(1)),
        abs(df['low'] - df['close'].shift(1))
    ], axis=1).max(axis=1)
    
    # Pine uses RMA (Wilder's) for ATR
    alpha = 1.0 / fast_atr_len
    orbATR_fast = tr.ewm(alpha=alpha, adjust=False).mean()
    
    # Slow = SMA of fast ATR
    orbATR_slow = orbATR_fast.rolling(10, min_periods=1).mean()
    
    # Session baseline (builds during 9:30-10:00)
    df['orb_window'] = df.index.map(lambda ts: 9*60+30 <= ts.hour*60+ts.minute < 10*60)
    df['new_day'] = df.index.date != pd.Series(df.index.date).shift(1).values
    
    # Build session baseline with exponential smoothing
    session_baseline = pd.Series(index=df.index, dtype=float)
    current_baseline = np.nan
    
    for i in range(len(df)):
        if df['new_day'].iloc[i]:
            current_baseline = np.nan
        
        if df['orb_window'].iloc[i]:
            if np.isnan(current_baseline):
                current_baseline = orbATR_slow.iloc[i]
            else:
                current_baseline = 0.9 * current_baseline + 0.1 * orbATR_slow.iloc[i]
        
        session_baseline.iloc[i] = current_baseline
    
    # Session VF
    sessionVF = np.where(
        (session_baseline > 0) & (~np.isnan(session_baseline)),
        orbATR_fast / session_baseline,
        1.0
    )
    
    # --- Component 2: Higher Timeframe (Daily) VF ---
    
    # Group by date and calculate daily ATR
    df['date'] = df.index.date
    
    # Daily high, low, close
    daily_data = df.groupby('date').agg({
        'high': 'max',
        'low': 'min',
        'close': 'last'
    })
    
    # Daily True Range
    daily_tr = pd.concat([
        daily_data['high'] - daily_data['low'],
        abs(daily_data['high'] - daily_data['close'].shift(1)),
        abs(daily_data['low'] - daily_data['close'].shift(1))
    ], axis=1).max(axis=1)
    
    # Daily ATR (14-period RMA)
    daily_atr = daily_tr.ewm(alpha=1/14, adjust=False).mean()
    
    # Daily ATR slow (20-period SMA)
    daily_atr_slow = daily_atr.rolling(20, min_periods=1).mean()
    
    # HTF VF
    htfVF_daily = np.where(
        (daily_atr_slow > 0) & (~np.isnan(daily_atr_slow)),
        daily_atr / daily_atr_slow,
        1.0
    )
    htfVF_daily = pd.Series(htfVF_daily, index=daily_data.index)
    
    # Map back to intraday
    df['htfVF'] = df['date'].map(htfVF_daily)
    
    # --- Final Vol Factor ---
    df['sessionVF'] = sessionVF
    df['volFactor'] = df['sessionVF'] * df['htfVF']
    
    # --- Classification ---
    def classify_vol(vf):
        if np.isnan(vf):
            return "NORMAL"
        elif vf < low_thresh:
            return "LOW"
        elif vf >= extreme_thresh:
            return "EXTREME"
        elif vf >= high_thresh:
            return "HIGH"
        else:
            return "NORMAL"
    
    df['volState'] = df['volFactor'].apply(classify_vol)
    
    return df


# Apply corrected calculation
df = calc_vol_state_pine(df)

# Test on 10/30 at 09:35
ts = '2025-10-30 09:35:00-04:00'
row = df.loc[ts]

print("=== CORRECTED Vol State Calculation ===")
print(f"\nSession VF: {row['sessionVF']:.3f}")
print(f"HTF VF:     {row['htfVF']:.3f}")
print(f"Vol Factor: {row['volFactor']:.3f}")
print(f"Vol State:  {row['volState']}")
print(f"\nThresholds: LOW < 0.8 | NORMAL | HIGH >= 1.3 | EXTREME >= 2.0")

if row['volState'] == 'EXTREME':
    print("\n✓ MATCHES Pine - EXTREME detected!")
else:
    print(f"\n✗ Still not matching - Pine shows EXTREME, Python shows {row['volState']}")

=== CORRECTED Vol State Calculation ===

Session VF: 2.029
HTF VF:     0.977
Vol Factor: 1.981
Vol State:  HIGH

Thresholds: LOW < 0.8 | NORMAL | HIGH >= 1.3 | EXTREME >= 2.0

✗ Still not matching - Pine shows EXTREME, Python shows HIGH


In [14]:
# Debug: Break down both components
ts = '2025-10-30 09:35:00-04:00'

print("=== Python Component Values ===")
print(f"Session VF: {df.loc[ts, 'sessionVF']:.3f}")
print(f"HTF VF:     {df.loc[ts, 'htfVF']:.3f}")
print(f"Product:    {df.loc[ts, 'volFactor']:.3f}")

print("\n=== Pine shows VF: 2.51 ===")
print(f"Gap: {2.51 - df.loc[ts, 'volFactor']:.3f}")

# If htfVF is similar, what would sessionVF need to be?
target_vf = 2.51
htf = df.loc[ts, 'htfVF']
needed_session_vf = target_vf / htf
print(f"\nTo get 2.51 with htfVF={htf:.3f}, sessionVF would need to be: {needed_session_vf:.3f}")
print(f"Python sessionVF is: {df.loc[ts, 'sessionVF']:.3f}")

=== Python Component Values ===
Session VF: 2.029
HTF VF:     0.977
Product:    1.981

=== Pine shows VF: 2.51 ===
Gap: 0.529

To get 2.51 with htfVF=0.977, sessionVF would need to be: 2.570
Python sessionVF is: 2.029


In [15]:
# Debug session VF components
ts = '2025-10-30 09:35:00-04:00'
idx = df.index.get_loc(ts)

# Recalculate components manually
tr = pd.concat([
    df['high'] - df['low'],
    abs(df['high'] - df['close'].shift(1)),
    abs(df['low'] - df['close'].shift(1))
], axis=1).max(axis=1)

# Fast ATR (5-period RMA)
fast_atr = tr.ewm(alpha=1/5, adjust=False).mean()

# Slow ATR (SMA of fast)
slow_atr = fast_atr.rolling(10, min_periods=1).mean()

print("=== Session VF Components at 09:35 ===")
print(f"Fast ATR (5):  {fast_atr.iloc[idx]:.4f}")
print(f"Slow ATR (10): {slow_atr.iloc[idx]:.4f}")

# What's the session baseline at this point?
print(f"\nSession Baseline: {df.loc[ts, 'sessionVF'] and fast_atr.iloc[idx] / df.loc[ts, 'sessionVF']:.4f}")

# Pine ATR was 0.645 - let's compare
print(f"\nPine ATR (14): ~0.645")
print(f"Python ATR (14): {df.loc[ts, 'atr']:.4f}")

# What if baseline is from prior days (stale)?
print("\n=== Check: Is baseline stale? ===")
orb_window_bars = df[(df.index.date == pd.Timestamp(ts).date()) & (df['orb_window'] == True)]
print(f"ORB window bars today: {len(orb_window_bars)}")
for t in orb_window_bars.index[:6]:
    print(f"  {t.strftime('%H:%M')} - Fast ATR: {fast_atr.loc[t]:.4f}")

=== Session VF Components at 09:35 ===
Fast ATR (5):  1.0141
Slow ATR (10): 0.7280

Session Baseline: 0.4999

Pine ATR (14): ~0.645
Python ATR (14): 0.7208

=== Check: Is baseline stale? ===
ORB window bars today: 30
  09:30 - Fast ATR: 0.9096
  09:31 - Fast ATR: 0.9017
  09:32 - Fast ATR: 0.9993
  09:33 - Fast ATR: 1.0115
  09:34 - Fast ATR: 1.0352
  09:35 - Fast ATR: 1.0141


In [16]:
# Trace baseline building for 10/30 morning
target_date = '2025-10-30'
morning = df[df.index.date == pd.Timestamp(target_date).date()]
morning = morning[morning.index.hour == 9]

# Recalc components
tr = pd.concat([
    df['high'] - df['low'],
    abs(df['high'] - df['close'].shift(1)),
    abs(df['low'] - df['close'].shift(1))
], axis=1).max(axis=1)

fast_atr = tr.ewm(alpha=1/5, adjust=False).mean()
slow_atr = fast_atr.rolling(10, min_periods=1).mean()

print("=== Baseline building on 10/30 ===")
print("Time     | Fast ATR | Slow ATR | In ORB Window?")
print("-" * 50)

baseline = np.nan
for ts in morning.index[:10]:
    in_orb = 9*60+30 <= ts.hour*60+ts.minute < 10*60
    fa = fast_atr.loc[ts]
    sa = slow_atr.loc[ts]
    
    if in_orb:
        if np.isnan(baseline):
            baseline = sa
        else:
            baseline = 0.9 * baseline + 0.1 * sa
    
    print(f"{ts.strftime('%H:%M')} | {fa:.4f}   | {sa:.4f}   | {'YES → baseline=' + f'{baseline:.4f}' if in_orb else 'no'}")

=== Baseline building on 10/30 ===
Time     | Fast ATR | Slow ATR | In ORB Window?
--------------------------------------------------
09:30 | 0.9096   | 0.4220   | YES → baseline=0.4220
09:31 | 0.9017   | 0.4744   | YES → baseline=0.4272
09:32 | 0.9993   | 0.5346   | YES → baseline=0.4380
09:33 | 1.0115   | 0.5961   | YES → baseline=0.4538
09:34 | 1.0352   | 0.6616   | YES → baseline=0.4746
09:35 | 1.0141   | 0.7280   | YES → baseline=0.4999
09:36 | 0.9733   | 0.7924   | YES → baseline=0.5291
09:37 | 0.9306   | 0.8533   | YES → baseline=0.5616
09:38 | 0.9365   | 0.9096   | YES → baseline=0.5964
09:39 | 0.9552   | 0.9667   | YES → baseline=0.6334


In [17]:
# Debug HTF (daily) component
target_date = pd.Timestamp('2025-10-30').date()

# Get daily data
daily = df.groupby(df.index.date).agg({
    'high': 'max',
    'low': 'min', 
    'close': 'last',
    'open': 'first'
})

# Calculate daily TR and ATR
daily['prev_close'] = daily['close'].shift(1)
daily['tr'] = pd.concat([
    daily['high'] - daily['low'],
    abs(daily['high'] - daily['prev_close']),
    abs(daily['low'] - daily['prev_close'])
], axis=1).max(axis=1)

daily['atr14'] = daily['tr'].ewm(alpha=1/14, adjust=False).mean()
daily['atr14_sma20'] = daily['atr14'].rolling(20, min_periods=1).mean()
daily['htfVF'] = daily['atr14'] / daily['atr14_sma20']

print("=== Daily ATR Data (last 10 days) ===")
print(daily[['tr', 'atr14', 'atr14_sma20', 'htfVF']].tail(10))

print(f"\n=== 10/30 specifically ===")
print(f"Daily ATR(14): {daily.loc[target_date, 'atr14']:.4f}")
print(f"Daily ATR SMA(20): {daily.loc[target_date, 'atr14_sma20']:.4f}")
print(f"HTF VF: {daily.loc[target_date, 'htfVF']:.3f}")

=== Daily ATR Data (last 10 days) ===
                tr     atr14  atr14_sma20     htfVF
2025-12-12  8.2000  7.033481     8.631032  0.814906
2025-12-15  3.4150  6.775018     8.480440  0.798899
2025-12-16  3.5900  6.547517     8.332069  0.785821
2025-12-17  7.1800  6.592694     8.194408  0.804536
2025-12-18  5.0100  6.479645     8.061741  0.803753
2025-12-19  7.5400  6.555384     7.907800  0.828977
2025-12-22  3.1100  6.309286     7.734426  0.815741
2025-12-23  6.4300  6.317908     7.571329  0.834452
2025-12-24  2.5216  6.046743     7.381818  0.819140
2025-12-26  4.6900  5.949833     7.204417  0.825859

=== 10/30 specifically ===
Daily ATR(14): 10.8381
Daily ATR SMA(20): 11.0982
HTF VF: 0.977


In [18]:
# Use PRIOR day's daily ATR (like Pine does)
target_date = pd.Timestamp('2025-10-30').date()

# Get the day BEFORE
prior_dates = [d for d in daily.index if d < target_date]
prior_date = prior_dates[-1] if prior_dates else None

print(f"Prior trading day: {prior_date}")
print(f"Prior Daily ATR(14): {daily.loc[prior_date, 'atr14']:.2f}")
print(f"Prior Daily ATR SMA(20): {daily.loc[prior_date, 'atr14_sma20']:.2f}")

corrected_htfVF = daily.loc[prior_date, 'atr14'] / daily.loc[prior_date, 'atr14_sma20']
print(f"Corrected htfVF: {corrected_htfVF:.3f}")

# Recalc total VF
sessionVF = 2.029  # from earlier
corrected_VF = sessionVF * corrected_htfVF
print(f"\nCorrected Vol Factor: {corrected_VF:.3f}")
print(f"Pine Vol Factor: 2.51")

Prior trading day: 2025-10-29
Prior Daily ATR(14): 11.22
Prior Daily ATR SMA(20): 11.23
Corrected htfVF: 0.999

Corrected Vol Factor: 2.027
Pine Vol Factor: 2.51


In [19]:
# How much data do we actually have before Oct 30?
oct30 = pd.Timestamp('2025-10-30').date()
days_before = [d for d in daily.index if d < oct30]
print(f"Trading days before Oct 30: {len(days_before)}")
print(f"First day in data: {daily.index[0]}")
print(f"Need ~34 days for ATR(14) + SMA(20) warmup")

Trading days before Oct 30: 2
First day in data: 2025-10-28
Need ~34 days for ATR(14) + SMA(20) warmup


In [20]:
# Re-fetch with more history
collector = PolygonDataCollector()
df = collector.fetch_bars('NVDA', days_back=120, bar_size=1)
print(f"✓ Loaded {len(df)} bars")
print(f"Date range: {df.index[0].date()} to {df.index[-1].date()}")

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching NVDA (1-min bars, 120 days)...
  ✓ Fetched 32095 bars, cached for next time
✓ Loaded 32095 bars
Date range: 2025-08-29 to 2025-12-26


In [21]:
# Re-apply Phase 1 (ORB detection) and vol state to new data
df = detect_orb_session(df)
df = calculate_orb_levels(df)
df['orb_complete'] = ~df['in_session'] & df['orb_high'].notna() & df['orb_low'].notna()
df = calc_atr(df)
df = calc_session_vwap(df)
df = calc_vol_state_pine(df)

# Now check Oct 30 again
ts = '2025-10-30 09:35:00-04:00'
row = df.loc[ts]

print("=== Vol State with proper warmup ===")
print(f"Session VF: {row['sessionVF']:.3f}")
print(f"HTF VF:     {row['htfVF']:.3f}")
print(f"Vol Factor: {row['volFactor']:.3f}")
print(f"Vol State:  {row['volState']}")
print(f"\nPine shows: VF 2.51 [EXTREME]")

=== Vol State with proper warmup ===
Session VF: 2.029
HTF VF:     1.102
Vol Factor: 2.235
Vol State:  EXTREME

Pine shows: VF 2.51 [EXTREME]


In [22]:
# Recalculate R:R with EXTREME vol state
ts = '2025-10-30 09:35:00-04:00'

# EXTREME vol state means rrDesired = max(2.0 - 0.8, 1.0) = 1.2
rr_desired = 1.2  # EXTREME adjustment

full_idx = df.index.get_loc(ts)
stops = calculate_all_stops(df, full_idx, is_long=False)
rr_results = calculate_best_rr(stops, is_long=False, rr_desired=rr_desired)

print("=== R:R with EXTREME vol adjustment ===")
for stop_type, data in rr_results.items():
    print(f"{stop_type}: R:R = {data['rr']:.2f}")

best_type, best_rr = select_best_stop(rr_results)
print(f"\nBest: {best_type} with R:R = {best_rr:.2f}")
print(f"Min Required: 1.5")
print(f"\n{'✓ PASSES' if best_rr >= 1.5 else '✗ FAILS - SKIP'}")
print(f"\nPine shows: SKIP 1.2:1 < 1.5 min")

=== R:R with EXTREME vol adjustment ===
atr_stop: R:R = 1.20
swing_stop: R:R = 0.57
vwap_stop: R:R = 1.07
hybrid_stop: R:R = 1.20

Best: atr_stop with R:R = 1.20
Min Required: 1.5

✗ FAILS - SKIP

Pine shows: SKIP 1.2:1 < 1.5 min


## Phase 2 Summary

Once validated:
- Stop calculations match Pine
- R:R calculation matches Pine
- Gate logic (1.5 minimum) working correctly

**Next:** Phase 3 - Confluence Scoring (SSL, WAE, QQE, Volume)